In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
try:
#     plt.style.use('belle2')
    # plt.style.use('belle2_serif')
    plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter


Welcome to JupyROOT 6.26/04


In [2]:
def cal_Br_Central_value(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec):
    print("Input values:")
    print("Nsig:", Nsig)
    print("Nref:", Nref)
    print("eff_sig:", eff_sig)
    print("eff_ref:", eff_ref)
    print("Br_ref_dec:", Br_ref_dec)
    
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    
    Br = Br_ref_dec * (N_real_sig / N_real_ref)
    
    return Br
    
from math import sqrt

def cal_Br_error_stat(Nsig_err, Nsig, Nref_err, Nref, central_value):
    Variance = (Nsig_err / Nsig)**2 + (Nref_err / Nref)**2
    TOTAL = sqrt(Variance) * central_value
    return TOTAL

def cal_Br_error_with_eff(Nsig_err, Nsig, Nref_err, Nref, eff_sig_err, eff_sig, eff_ref_err, eff_ref, central_value):
    Variance = (Nsig_err / Nsig)**2 + (eff_sig_err / eff_sig)**2 + (Nref_err / Nref)**2 + (eff_ref_err / eff_ref)**2
    TOTAL = sqrt(Variance) * central_value
    return TOTAL


In [3]:
from math import sqrt

# Function to calculate the central branching ratio value
def calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec):
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    Br = Br_ref_dec * (N_real_sig / N_real_ref)
    return Br
    
def calculate_br_ratio(Nsig,Nsig_err,  Nref,Nref_err, eff_sig, eff_ref):
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    Br_ratio = (N_real_sig / N_real_ref)
    Br_ratio_err = (eff_ref/eff_sig) * math.sqrt( (Nsig_err / Nref)**2 + (Nsig / (Nref**2) * Nref_err)**2 )
    print(f'Br_ratio value = {Br_ratio:.4e}')
    print(f'Br_ratio Statistical uncertainty = {Br_ratio_err:.4e}')
    return Br_ratio, Br_ratio_err

# Function to calculate statistical uncertainty
def calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value):
    variance = (Nsig_err / Nsig)**2 + (Nref_err / Nref)**2
    return sqrt(variance) * central_value

# Function to print results in a formatted way
def print_results(label, central_value, stat_unc, Br_sig_dec):
    pull = (central_value - Br_sig_dec) / stat_unc
    print(f'{label} Central value = {central_value:.4e}')
    print(f'{label} Statistical uncertainty = {stat_unc:.4e}')
    print(f'{label} Pull = {pull:.4f}')
    print(f'{label} stas. unc./Central value = {stat_unc/central_value:.4e}\n')

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    return central_value, error

In [4]:
# signal_eff_error = math.sqrt(signal_eff * (1 - signal_eff) / N_gen)

def calculate_sig_eff_err(eff, N_gen):

    error = math.sqrt(eff * (1 - eff) / N_gen)
    return error

# Br(D+ -> eta K+)

In [5]:
# # Constants
# Br_ref_dec = 0.003770000
# Br_sig_dec = 0.000125000

# Br_sig_PDG = 0.0001250000
# Br_sig_PDG_err = 0.0000160000


In [6]:
Br_ref_pdg_2026 = 3.75 * 10**(-3)
Br_sig_pdg_2026 = 1.16 * 10**(-4)
Br_sig_PDG_err = 0.09 * 10**(-4)

In [7]:
Br_sig_pdg_2026/Br_ref_pdg_2026

0.030933333333333334

In [8]:
#fitv12(same to fitv15)
eff_sig_cal =  0.055650
eff_ref_cal =  0.076356

In [9]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)

eff_sig_err_cal


eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)
eff_ref_err_cal

print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 9.358871e-05, ref eff error: 1.084172e-04


In [10]:
# First calculation for mode: eta -> gg
Nsig_err, Nsig, Nref_err, Nref = 76.02191541771788, 737.4355712087948 , 300.754617677012 , 35361.130327815525
# Nsig_err, Nsig, Nref_err, Nref = , , ,

eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal, eff_ref_err_cal, eff_ref_cal

central_value_1 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_pdg_2026)
stat_unc_1 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_1)
print_results("Mode: eta -> gg", central_value_1, stat_unc_1, Br_sig_pdg_2026)


Mode: eta -> gg Central value = 1.0730e-04
Mode: eta -> gg Statistical uncertainty = 1.1099e-05
Mode: eta -> gg Pull = -0.7837
Mode: eta -> gg stas. unc./Central value = 1.0344e-01



In [11]:
ratio_central_Br_gg, ratio_err_Br_gg  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 2.8614e-02
Br_ratio Statistical uncertainty = 2.9598e-03


In [12]:
#fitv12(same to fitv15)
eff_sig_cal = 0.056526
eff_ref_cal = 0.075213

In [13]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)

eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)

print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 9.427867e-05, ref eff error: 1.076693e-04


In [14]:
# Second calculation for mode: eta -> pipipi
Nsig_err, Nsig, Nref_err, Nref = 44.15347155346362, 439.48631531165574, 181.61538396351716, 19966.42690530978
# Nsig_err, Nsig, Nref_err, Nref =, , ,

eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal ,eff_ref_err_cal,eff_ref_cal

central_value_2 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_pdg_2026)
stat_unc_2 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_2)
print_results("Mode: eta -> pipipi", central_value_2, stat_unc_2, Br_sig_pdg_2026)

Mode: eta -> pipipi Central value = 1.0983e-04
Mode: eta -> pipipi Statistical uncertainty = 1.1079e-05
Mode: eta -> pipipi Pull = -0.5569
Mode: eta -> pipipi stas. unc./Central value = 1.0088e-01



In [15]:
ratio_central_Br_3pi, ratio_err_Br_3pi  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 2.9288e-02
Br_ratio Statistical uncertainty = 2.9545e-03


In [16]:
# Combined error-weighted result of absoulte Br
combined_central_value, combined_error = combine_error_weighted(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print_results("Combined", combined_central_value, combined_error, Br_sig_pdg_2026)

Combined Central value = 1.0857e-04
Combined Statistical uncertainty = 7.8413e-06
Combined Pull = -0.9478
Combined stas. unc./Central value = 7.2225e-02



In [17]:
# Combined error-weighted result of Br ratio
combined_central_value, combined_error = combine_error_weighted(ratio_central_Br_gg,ratio_central_Br_3pi, ratio_err_Br_gg, ratio_err_Br_3pi)
print_results("Combined", combined_central_value, combined_error, Br_sig_pdg_2026/Br_ref_pdg_2026)

Combined Central value = 2.8952e-02
Combined Statistical uncertainty = 2.0910e-03
Combined Pull = -0.9478
Combined stas. unc./Central value = 7.2225e-02



In [18]:
# 2.895 \pm 0.209

# Br(Ds+ -> eta K+)

In [46]:
# Constants
# Br_ref_dec = 0.017000000
# Br_sig_dec = 0.001600000

# Br_sig_PDG = 0.001730000
# Br_sig_PDG_err = 0.000080000


In [47]:
Br_ref_pdg_2026 = 1.686 * 10**(-2)
Br_sig_pdg_2026 = 1.76 * 10**(-3)
Br_sig_PDG_err = 0.08 * 10**(-3)

In [48]:
Br_sig_pdg_2026/Br_ref_pdg_2026

0.10438908659549229

In [49]:
#fitv12 (same for fitv15)
eff_sig_cal = 0.048909
eff_ref_cal = 0.068025

In [50]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal,  6e+6)
eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)
print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 8.805009e-05, ref eff error: 1.027923e-04


In [51]:
# First calculation for mode: eta -> gg
Nsig_err, Nsig, Nref_err, Nref = 105.33704702697923, 3781.194318722153 ,287.5754911215081, 55606.091428310785
# Nsig_err, Nsig, Nref_err, Nref =  , , ,

eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal, eff_ref_err_cal, eff_ref_cal

central_value_1 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_pdg_2026)
stat_unc_1 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_1)
print_results("Mode: eta -> gg", central_value_1, stat_unc_1, Br_sig_pdg_2026)

Mode: eta -> gg Central value = 1.5946e-03
Mode: eta -> gg Statistical uncertainty = 4.5181e-05
Mode: eta -> gg Pull = -3.6615
Mode: eta -> gg stas. unc./Central value = 2.8334e-02



In [52]:
ratio_central_Br_gg, ratio_err_Br_gg  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 9.4577e-02
Br_ratio Statistical uncertainty = 2.6798e-03


In [53]:
#fitv12 (same for fitv15)# fitv12
eff_sig_cal  = 0.049451
eff_ref_cal =  0.067130

In [54]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)
eff_sig_err_cal
# eff_ref_err_cal = calculate_sig_eff_err(0.02653, 2e+6)
eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)

print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 8.851139e-05, ref eff error: 1.021629e-04


In [55]:
# Second calculation for mode: eta -> pipipi
Nsig_err, Nsig, Nref_err, Nref = 61.13351651463461, 1908.9868838078783, 197.38980648903453, 30862.391056543824
# Nsig_err, Nsig, Nref_err, Nref = 61.550459514734825, 1945.0360260582765, 197.38980648903453, 30862.391056543824

# Nsig_err, Nsig, Nref_err, Nref = , , ,
eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal , eff_ref_err_cal, eff_ref_cal

central_value_2 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_pdg_2026)
stat_unc_2 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_2)
print_results("Mode: eta -> pipipi", central_value_2, stat_unc_2, Br_sig_pdg_2026)

Mode: eta -> pipipi Central value = 1.4157e-03
Mode: eta -> pipipi Statistical uncertainty = 4.6232e-05
Mode: eta -> pipipi Pull = -7.4471
Mode: eta -> pipipi stas. unc./Central value = 3.2656e-02



In [56]:
ratio_central_Br_3pi, ratio_err_Br_3pi  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 8.3968e-02
Br_ratio Statistical uncertainty = 2.7421e-03


In [57]:
# Combined error-weighted result
combined_central_value, combined_error = combine_error_weighted(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print_results("Combined", combined_central_value, combined_error, Br_sig_pdg_2026)

Combined Central value = 1.5072e-03
Combined Statistical uncertainty = 3.2313e-05
Combined Pull = -7.8237
Combined stas. unc./Central value = 2.1439e-02



In [58]:
# Combined error-weighted result
combined_central_value, combined_error = combine_error_weighted(ratio_central_Br_gg,ratio_central_Br_3pi, ratio_err_Br_gg, ratio_err_Br_3pi)
print_results("Combined", combined_central_value, combined_error,  Br_sig_pdg_2026/Br_ref_pdg_2026)

Combined Central value = 8.9395e-02
Combined Statistical uncertainty = 1.9165e-03
Combined Pull = -7.8237
Combined stas. unc./Central value = 2.1439e-02



In [59]:
# Mine ratio: 8.94 \pm 0.19 \pm 0.151

In [ ]:
# BESIII ratio: 9.31 \pm 0.58 \pm 0.10
# CELO ratio: 8.9 \pm 1.5 \pm 0.4

In [17]:
Br_sig_pdg_2026/Br_ref_pdg_2026


0.10438908659549229

In [18]:
(1.76 - 1.5073 ) / 0.032

7.896874999999998

In [19]:
1.5073*0.0160142348

0.02413825611404

In [20]:
# 1.507 \pm 0.032 \pm 0.02

In [ ]:
# PDG 2026 1.76 \pm 0.08

In [21]:
# PDG 2020 1.72±0.34